# Local SME Underwriting Agents Notebook

In [ ]:
from __future__ import annotations

import hashlib
import httpx
import json
import os
import re
import shutil
import sys
import threading
import time
import unicodedata

from dataclasses import asdict, dataclass, field
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Any, Literal, TypedDict

from dotenv import load_dotenv


PROJECT_ROOT = Path().resolve()
load_dotenv(PROJECT_ROOT / ".env", override=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
TESTCASE_ID = "case_1"

# Chọn trên màn hình upload. Để rỗng thì dùng DEFAULT_LOAN_PROGRAM (PLO).
# Một trong: B1CP, MISA, PLPP, PLO
LOAN_PROGRAM = "PLO"

# Agent chọn trên màn hình upload. Một trong: BUSINESS_ACTIVITY_AGENT,
# FINANCIAL_ANALYSIS_AGENT, CREDIT_RELATIONSHIP_AGENT, CREDIT_PROPOSAL_AGENT
AGENT = "BUSINESS_ACTIVITY_AGENT"

# CSV/MD/PDF/TXT/XLS/XLSX/XML
INPUT_PATHS = [
    str(PROJECT_ROOT / "testing" / "samples" / TESTCASE_ID),
    # "/absolute/path/to/BCTC.pdf",
]

OUTPUT_DIR = PROJECT_ROOT / "logs"
MAX_CHARS_PER_DOCUMENT = 120_000

In [ ]:
from src.config import Config, build_llm

config = Config(
    document_llm=build_llm(
        "MODEL_DOCUMENT",
        temperature=0.5
    ),
    analysis_llm=build_llm(
        "MODEL_ANALYZER",
        temperature=0.1,
    ),
    financial_statement_extraction_llm=build_llm(
        "MODEL_BCTC_EXTRACTION",
        temperature=0.0,
    ),
    proposal_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    cic_s10a_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    cic_r21_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    sitevisit_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)

In [ ]:
from src.types import (
    AgentName,
    WorkflowMode,
    DocumentAgentName,
    ClassifiedDocument,
    UnderwritingGraphState,
    to_dict_list,
    extract_text_from_agent_output,
    truncate_text,
)

In [ ]:
from src.agents.documents.document_classification import (
    document_type_scores,
    rule_classify_document,
)
from src.agents.documents.document_discovery import (
    compute_file_hash,
    discover_documents,
    resolve_input_path,
)
from src.utils.common import (
    SUPPORTED_EXTENSIONS,
    normalize_text,
)
from src.types import VALID_DOCUMENT_AGENTS
from src.matrix.document_matrix import (
    agent_relevance_for_type,
    all_types,
    load_matrix,
    primary_agent_for_type,
)

_matrix = load_matrix()
print(
    f"Document matrix v{_matrix.version}: {len(all_types())} document types \n"
    f"Loan programs: {', '.join(_matrix.loan_programs)}"
)


In [ ]:
from src.agents.specialist import (
    SpecialistAgent,
    BusinessActivityAnalysis,
    FinancialAnalysis,
    CreditRelationshipAnalysis,
    CreditProposalAnalysis,
)
from src.agents.supervisor import Supervisor

In [ ]:
from IPython.display import Markdown, display
from src.utils.common import show_graph

supervisor = Supervisor(config)

display(Markdown("### Workflow Graph"))
display(show_graph(supervisor.workflow_graph))

result = supervisor.process(
    INPUT_PATHS, agent=AGENT, loan_program=LOAN_PROGRAM
)

display(Markdown(result["response"]))

print("\n--- Agent Name ---")
print(result["agent_name"])

print("\n--- Steps ---")
for step in result["steps"]:
    print("-", step)

print("\n--- Document Classifications ---")
classification_keys = [
    "filename",
    "document_type",
    "declared_group",
    "document_group",
    "source_description",
    "agent_relevance",
    "loan_program",
    "agent",
    "confidence",
    "reasoning",
    "extraction_status",
    "extraction_error",
    "is_financial_statement",
    "financial_statement_extraction_error",
    "is_proposal",
    "proposal_extraction_error",
    "is_cic_s10a",
    "cic_s10a_extraction_error",
    "is_cic_r21",
    "cic_r21_extraction_error",
    "classifier_error_type",
    "classifier_error",
]
for item in result["document_classifications"]:
    classification = {key: item.get(key) for key in classification_keys}
    print(json.dumps(classification, ensure_ascii=False, indent=2))

_extraction_failures = [
    (item["filename"], label, item.get(f"{prefix}_extraction_error") or "?")
    for item in result["document_classifications"]
    for prefix, flag, label in (
        ("bctc", "is_financial_statement", "BCTC"),
        ("proposal", "is_proposal", "đề nghị cấp tín dụng"),
        ("cic_s10a", "is_cic_s10a", "CIC S10A"),
        ("cic_r21", "is_cic_r21", "CIC R21"),
    )
    if item.get(flag) and not item.get(f"{prefix}_extraction")
]
if _extraction_failures:
    print(f"\nWARNING: {len(_extraction_failures)} lần trích xuất thất bại — "
          "agent sẽ dùng OCR thô thay cho dữ liệu có cấu trúc:")
    for _name, _label, _why in _extraction_failures:
        print(f"  - {_name} ({_label}): {_why}")

_unmatched = [
    item["filename"]
    for item in result["document_classifications"]
    if not item.get("document_type")
]
if _unmatched:
    print(
        f"\nWARNING: {len(_unmatched)} document(s) matched no type in the "
        f"matrix and were shared with every agent: {', '.join(_unmatched)}"
    )


In [ ]:
run_id = "_".join([TESTCASE_ID, datetime.now().strftime("%Y%m%d_%H%M%S")])
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

from src.utils.report_html import embed_diagrams

post_monitor_content = [
    {
        'filename': 'result.json',
        'content': result},
    {
        'filename': 'document_classifications.json',
        'content': result["document_classifications"]},
    {
        'filename': 'document_selections.json',
        'content': result["document_selections"]},
    {
        'filename': 'financial_metrics.json',
        'content': result["financial_metrics"]},
    {
        'filename': 'credit_need.json',
        'content': result["credit_need"]},
    {
        'filename': 'agent_outputs.json',
        'content': result["sub_agent_outputs"]},
    {
        'filename': 'financial_metrics.json',
        'content': result.get("financial_metrics", {})},
    {
        'filename': 'financial_statement_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'financial_statement_extraction',
        'condition': 'is_financial_statement' 
    },
    {
        'filename': 'proposal_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'proposal_extraction',
        'condition': 'is_proposal' 
    },
    {
        'filename': 'cic_s10a_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'cic_s10a_extraction',
        'condition': 'is_cic_s10a' 
    },
    {
        'filename': 'cic_r21_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'cic_r21_extraction',
        'condition': 'is_cic_r21' 
    },
    {
        'filename': 'sitevisit_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'sitevisit_extraction',
        'condition': 'is_sitevisit' 
    }
]

(run_dir / "final_response.md").write_text(
    embed_diagrams(result["response"]), 
    encoding="utf-8"
)

for c in post_monitor_content:
    if c.get('content_lv2', None):
        (run_dir / "proposal_extraction.json").write_text(
            json.dumps(
                {
                    doc["filename"]: doc.get(c['content_lv2'])
                    for doc in c['content']
                    if doc.get(c['condition'])
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
    else:
        (run_dir / c['filename']).write_text(
            json.dumps(c['content'], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

# Export the final response (Markdown) to PDF via `markdown` + WeasyPrint.
import platform

if platform.system() == "Darwin":
    for brew_lib in ("/opt/homebrew/lib", "/usr/local/lib"):
        if os.path.isdir(brew_lib):
            os.environ["DYLD_LIBRARY_PATH"] = (
                brew_lib + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
            )

from weasyprint import HTML
from src.utils.report_html import build_report_html

html_doc = build_report_html(result["response"])
HTML(string=html_doc).write_pdf(str(run_dir / "final_response.pdf"))

print(f"Saved PDF: {run_dir / 'final_response.pdf'}")
print(f"Saved artifacts to: {run_dir}")